# Predicción de abandono de clientes de Beta Bank

## Objetivo del proyecto

Beta Bank ha detectado una pérdida progresiva de clientes. Debido a que retener
a los clientes existentes resulta más económico que captar nuevos, el banco
necesita identificar con anticipación qué clientes presentan mayor probabilidad
de abandonar la entidad.

El objetivo de este proyecto es desarrollar un modelo de clasificación capaz de
predecir si un cliente abandonará el banco a partir de sus características y de
su comportamiento histórico.

La variable objetivo es `Exited`:

- `0`: el cliente permanece en el banco.
- `1`: el cliente abandona el banco.

La métrica principal de evaluación será **F1**, ya que permite evaluar de forma
conjunta la precisión y el recall de la clase positiva. El modelo final deberá
obtener un valor **F1 de al menos 0.59 en el conjunto de prueba**.

También se calculará la métrica **AUC-ROC** para evaluar la capacidad del modelo
para distinguir entre clientes que abandonan y clientes que permanecen a través
de distintos umbrales de clasificación.

In [ ]:
import pandas as pd

data = pd.read_csv('/datasets/Churn.csv')

print('Tamaño del dataset:')
print(data.shape)

print('\nPrimeras cinco filas:')
display(data.head())

print('\nInformación general:')
data.info()

print('\nEstadísticas descriptivas:')
display(data.describe())

print('\nValores ausentes:')
print(data.isna().sum())

print('\nDuplicados:')
print(data.duplicated().sum())

print('\nDistribución de Exited:')
print(data['Exited'].value_counts())

print('\nDistribución relativa de Exited:')
print(data['Exited'].value_counts(normalize=True))

Tamaño del dataset:
(10000, 14)

Primeras cinco filas:


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2.0,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1.0,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8.0,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1.0,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2.0,125510.82,1,1,1,79084.10,0



Información general:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           9091 non-null   float64
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(3), int64(8), object(3)
memory usage: 1.1+ MB

Estadísticas descriptivas:


,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.00000,1.000000e+04,10000.000000,10000.000000,9091.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,5000.50000,1.569094e+07,650.528800,38.921800,4.997690,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,2886.89568,7.193619e+04,96.653299,10.487806,2.894723,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.00000,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2500.75000,1.562853e+07,584.000000,32.000000,2.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,5000.50000,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,7500.25000,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,10000.00000,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000



Valores ausentes:
RowNumber            0
CustomerId           0
Surname              0
CreditScore          0
Geography            0
Gender               0
Age                  0
Tenure             909
Balance              0
NumOfProducts        0
HasCrCard            0
IsActiveMember       0
EstimatedSalary      0
Exited               0
dtype: int64

Duplicados:
0

Distribución de Exited:
0    7963
1    2037
Name: Exited, dtype: int64

Distribución relativa de Exited:
0    0.7963
1    0.2037
Name: Exited, dtype: float64


### Conclusiones de la exploración inicial

El conjunto de datos contiene 10,000 observaciones y 14 columnas.

No se encontraron filas duplicadas. La única característica con valores ausentes
es `Tenure`, con 909 valores faltantes, aproximadamente el 9.1% de las
observaciones. Debido a que eliminar estas filas supondría una pérdida
considerable de información, los valores ausentes serán imputados durante el
preprocesamiento.

Las columnas `RowNumber`, `CustomerId` y `Surname` no representan
características útiles del comportamiento financiero del cliente y serán
eliminadas antes del entrenamiento.

Las características `Geography` y `Gender` son categóricas y deberán
transformarse a valores numéricos.

La variable objetivo `Exited` presenta un desequilibrio de clases:
aproximadamente el 79.6% de los clientes permanece en el banco y el 20.4%
abandona.

Debido a este desequilibrio, la exactitud por sí sola podría ofrecer una visión
engañosa del rendimiento del modelo. Por este motivo, F1 será la métrica
principal y también se utilizará AUC-ROC como métrica complementaria.

## Preparación de los datos

### Eliminación de características irrelevantes

`RowNumber`, `CustomerId` y `Surname` se eliminan porque funcionan como
identificadores o información personal y no representan directamente el
comportamiento financiero del cliente.

In [ ]:
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

display(data.head())
print(data.shape)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2.0,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1.0,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8.0,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1.0,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2.0,125510.82,1,1,1,79084.10,0


(10000, 11)


In [ ]:
print(data['Tenure'].describe())
print()
print(data['Tenure'].value_counts().sort_index())

count    9091.000000
mean        4.997690
std         2.894723
min         0.000000
25%         2.000000
50%         5.000000
75%         7.000000
max        10.000000
Name: Tenure, dtype: float64

0.0     382
1.0     952
2.0     950
3.0     928
4.0     885
5.0     927
6.0     881
7.0     925
8.0     933
9.0     882
10.0    446
Name: Tenure, dtype: int64


### División de los datos

Los datos se dividirán en tres conjuntos:

- 60% entrenamiento;
- 20% validación;
- 20% prueba.

Se utilizará estratificación para conservar aproximadamente la proporción de
las clases de `Exited` en los tres conjuntos.

In [ ]:
target = data['Exited']
features = data.drop('Exited', axis=1)

from sklearn.model_selection import train_test_split

features_train, features_temp, target_train, target_temp = train_test_split(
    features,
    target,
    test_size=0.40,
    random_state=12345,
    stratify=target
)

features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp,
    target_temp,
    test_size=0.50,
    random_state=12345,
    stratify=target_temp
)

In [ ]:
print('Train:', features_train.shape, target_train.shape)
print('Valid:', features_valid.shape, target_valid.shape)
print('Test:', features_test.shape, target_test.shape)

print('\nProporción de clases en train:')
print(target_train.value_counts(normalize=True))

print('\nProporción de clases en valid:')
print(target_valid.value_counts(normalize=True))

print('\nProporción de clases en test:')
print(target_test.value_counts(normalize=True))

Train: (6000, 10) (6000,)
Valid: (2000, 10) (2000,)
Test: (2000, 10) (2000,)

Proporción de clases en train:
0    0.796333
1    0.203667
Name: Exited, dtype: float64

Proporción de clases en valid:
0    0.796
1    0.204
Name: Exited, dtype: float64

Proporción de clases en test:
0    0.7965
1    0.2035
Name: Exited, dtype: float64


La división conserva aproximadamente la distribución original de las clases en
los tres conjuntos. Esto confirma que la estratificación se realizó
correctamente.

### Tratamiento de valores ausentes

La mediana de `Tenure` se calcula únicamente utilizando el conjunto de
entrenamiento. Ese mismo valor se utiliza posteriormente para rellenar los
valores ausentes de entrenamiento, validación y prueba, evitando utilizar
información de los conjuntos de evaluación durante el preprocesamiento.

In [ ]:
tenure_median = features_train['Tenure'].median()

print(tenure_median)

5.0


In [ ]:
features_train = features_train.copy()
features_valid = features_valid.copy()
features_test = features_test.copy()

features_train.loc[:, 'Tenure'] = features_train['Tenure'].fillna(tenure_median)
features_valid.loc[:, 'Tenure'] = features_valid['Tenure'].fillna(tenure_median)
features_test.loc[:, 'Tenure'] = features_test['Tenure'].fillna(tenure_median)

print(features_train['Tenure'].isna().sum())
print(features_valid['Tenure'].isna().sum())
print(features_test['Tenure'].isna().sum())

0
0
0


### Codificación de variables categóricas

Las variables categóricas `Geography` y `Gender` se transformarán mediante
One-Hot Encoding para que puedan ser utilizadas por los modelos.

Se utilizará `drop_first=True` para evitar columnas redundantes.

In [ ]:
features_train = pd.get_dummies(features_train, drop_first=True)
features_valid = pd.get_dummies(features_valid, drop_first=True)
features_test = pd.get_dummies(features_test, drop_first=True)

print(features_train.shape)
print(features_valid.shape)
print(features_test.shape)

print(features_train.columns)

(6000, 11)
(2000, 11)
(2000, 11)
Index(['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Geography_Germany',
       'Geography_Spain', 'Gender_Male'],
      dtype='object')


In [ ]:
print(features_train.columns.equals(features_valid.columns))
print(features_train.columns.equals(features_test.columns))

True
True


Las columnas resultantes son iguales en los conjuntos de entrenamiento,
validación y prueba, por lo que la codificación es consistente entre los tres.

### Estandarización de características numéricas

Las características numéricas se estandarizan utilizando `StandardScaler`.

El escalador se ajusta exclusivamente con el conjunto de entrenamiento para
evitar utilizar información de validación o prueba durante el entrenamiento.

La estandarización es especialmente importante para la regresión logística.
Los modelos basados en árboles no la requieren, aunque pueden trabajar sin
problemas con estas características transformadas.

In [ ]:
from sklearn.preprocessing import StandardScaler

numeric = [
    'CreditScore',
    'Age',
    'Tenure',
    'Balance',
    'NumOfProducts',
    'EstimatedSalary'
]

scaler = StandardScaler()

scaler.fit(features_train[numeric])

features_train[numeric] = scaler.transform(features_train[numeric])
features_valid[numeric] = scaler.transform(features_valid[numeric])
features_test[numeric] = scaler.transform(features_test[numeric])

In [ ]:
display(features_train.head())

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_Germany,Geography_Spain,Gender_Male
2837,-1.040434,0.953312,0.360600,0.774657,-0.914708,0,1,-0.119110,1,0,0
9925,0.454006,-0.095244,-0.002786,1.910540,-0.914708,1,1,-0.258658,0,0,0
8746,0.103585,-0.476537,1.087371,0.481608,0.820981,0,1,1.422836,0,0,1
660,-0.184996,0.190726,-0.002786,0.088439,-0.914708,1,1,-1.160427,1,0,0
3610,-0.720933,1.620574,-1.456328,0.879129,-0.914708,1,0,0.113236,0,0,0


## Modelado

### Modelo base sin corrección del desequilibrio

Primero se entrenará una regresión logística sin aplicar ninguna técnica para
corregir el desequilibrio de clases.

Este modelo servirá como referencia para comprobar posteriormente si las
técnicas de balanceo mejoran el rendimiento.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

model = LogisticRegression(
    random_state=12345,
    solver='liblinear'
)

model.fit(features_train, target_train)

predicted_valid = model.predict(features_valid)

probabilities_valid = model.predict_proba(features_valid)
probabilities_one_valid = probabilities_valid[:, 1]

f1 = f1_score(target_valid, predicted_valid)
auc_roc = roc_auc_score(target_valid, probabilities_one_valid)

print('F1:', f1)
print('AUC-ROC:', auc_roc)

F1: 0.3076923076923077
AUC-ROC: 0.7875052345058627


### Resultados del modelo base

La regresión logística sin corrección del desequilibrio obtuvo aproximadamente:

- F1 = 0.308
- AUC-ROC = 0.788

El valor F1 está considerablemente por debajo del objetivo mínimo de 0.59, lo
que indica que el modelo presenta dificultades para clasificar adecuadamente
la clase positiva (`Exited = 1`).

Sin embargo, el AUC-ROC es claramente superior a 0.5, por lo que el modelo
presenta cierta capacidad para distinguir entre clientes que abandonan y
clientes que permanecen.

A continuación se probarán técnicas para tratar el desequilibrio de clases y
comprobar si el rendimiento mejora.

## Tratamiento del desequilibrio de clases

### Regresión logística con pesos de clase

In [ ]:
model_balanced = LogisticRegression(
    random_state=12345,
    solver='liblinear',
    class_weight='balanced'
)

model_balanced.fit(features_train, target_train)

predicted_valid_balanced = model_balanced.predict(features_valid)

probabilities_valid_balanced = model_balanced.predict_proba(features_valid)
probabilities_one_valid_balanced = probabilities_valid_balanced[:, 1]

f1_balanced = f1_score(
    target_valid,
    predicted_valid_balanced
)

auc_roc_balanced = roc_auc_score(
    target_valid,
    probabilities_one_valid_balanced
)

print('F1 balanceado:', f1_balanced)
print('AUC-ROC balanceado:', auc_roc_balanced)

F1 balanceado: 0.5289982425307558
AUC-ROC balanceado: 0.7937004261503596


### Resultados con pesos de clase

La regresión logística utilizando `class_weight='balanced'` obtuvo
aproximadamente:

- F1 = 0.529
- AUC-ROC = 0.794

El F1 mejoró considerablemente respecto al modelo base, que obtuvo
aproximadamente 0.308. Esto muestra que dar mayor importancia a la clase
minoritaria mejora la capacidad del modelo para identificar clientes que
abandonan el banco.

El AUC-ROC también presentó una ligera mejora.

Sin embargo, el F1 continúa por debajo del objetivo mínimo de 0.59, por lo que
se probará una segunda técnica de balanceo.

### Sobremuestreo de la clase minoritaria

Como segunda estrategia para tratar el desequilibrio, se aplicará
sobremuestreo.

La idea consiste en repetir observaciones de la clase minoritaria
`Exited = 1` dentro del conjunto de entrenamiento.

El conjunto de validación y el conjunto de prueba no se modifican.

In [ ]:
from sklearn.utils import shuffle

def upsample(features, target, repeat):
    features_zeros = features[target == 0]
    features_ones = features[target == 1]

    target_zeros = target[target == 0]
    target_ones = target[target == 1]

    features_upsampled = pd.concat(
        [features_zeros] + [features_ones] * repeat
    )

    target_upsampled = pd.concat(
        [target_zeros] + [target_ones] * repeat
    )

    features_upsampled, target_upsampled = shuffle(
        features_upsampled,
        target_upsampled,
        random_state=12345
    )

    return features_upsampled, target_upsampled

In [ ]:
features_upsampled, target_upsampled = upsample(
    features_train,
    target_train,
    4
)

print(features_upsampled.shape)
print(target_upsampled.shape)

print(target_upsampled.value_counts())
print(target_upsampled.value_counts(normalize=True))

(9666, 11)
(9666,)
1    4888
0    4778
Name: Exited, dtype: int64
1    0.50569
0    0.49431
Name: Exited, dtype: float64


Después del sobremuestreo, las dos clases quedan aproximadamente equilibradas
en el conjunto de entrenamiento.

In [ ]:
model_upsampled = LogisticRegression(
    random_state=12345,
    solver='liblinear'
)

model_upsampled.fit(
    features_upsampled,
    target_upsampled
)

predicted_valid_upsampled = model_upsampled.predict(features_valid)

probabilities_valid_upsampled = model_upsampled.predict_proba(features_valid)
probabilities_one_valid_upsampled = probabilities_valid_upsampled[:, 1]

f1_upsampled = f1_score(
    target_valid,
    predicted_valid_upsampled
)

auc_roc_upsampled = roc_auc_score(
    target_valid,
    probabilities_one_valid_upsampled
)

print('F1 con sobremuestreo:', f1_upsampled)
print('AUC-ROC con sobremuestreo:', auc_roc_upsampled)

F1 con sobremuestreo: 0.5246753246753246
AUC-ROC con sobremuestreo: 0.7937774041777517


### Resultados con sobremuestreo

La regresión logística entrenada con los datos sobremuestreados obtuvo
aproximadamente:

- F1 = 0.525
- AUC-ROC = 0.794

El sobremuestreo mejoró considerablemente el F1 respecto al modelo base, aunque
el resultado fue ligeramente inferior al obtenido con
`class_weight='balanced'`.

Las dos estrategias de balanceo mejoraron la regresión logística, pero ninguna
alcanzó el objetivo mínimo de F1 = 0.59.

Por este motivo, a continuación se evaluarán modelos basados en árboles.

## Modelos basados en árboles

### Árbol de decisión

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(
    random_state=12345
)

tree_model.fit(features_train, target_train)

tree_predicted_valid = tree_model.predict(features_valid)

tree_probabilities_valid = tree_model.predict_proba(features_valid)
tree_probabilities_one_valid = tree_probabilities_valid[:, 1]

tree_f1 = f1_score(
    target_valid,
    tree_predicted_valid
)

tree_auc_roc = roc_auc_score(
    target_valid,
    tree_probabilities_one_valid
)

print('F1 árbol base:', tree_f1)
print('AUC-ROC árbol base:', tree_auc_roc)

F1 árbol base: 0.5035971223021583
AUC-ROC árbol base: 0.6895137451965709


In [ ]:
best_depth = 0
best_f1 = 0

for depth in range(1, 21):
    model = DecisionTreeClassifier(
        random_state=12345,
        max_depth=depth
    )

    model.fit(features_train, target_train)

    predicted_valid = model.predict(features_valid)

    f1 = f1_score(
        target_valid,
        predicted_valid
    )

    print('Profundidad:', depth, '| F1:', f1)

    if f1 > best_f1:
        best_f1 = f1
        best_depth = depth

print()
print('Mejor profundidad:', best_depth)
print('Mejor F1:', best_f1)

Profundidad: 1 | F1: 0.0
Profundidad: 2 | F1: 0.5203488372093023
Profundidad: 3 | F1: 0.5375722543352601
Profundidad: 4 | F1: 0.5206349206349208
Profundidad: 5 | F1: 0.5454545454545455
Profundidad: 6 | F1: 0.5787878787878787
Profundidad: 7 | F1: 0.5924855491329479
Profundidad: 8 | F1: 0.5601217656012176
Profundidad: 9 | F1: 0.5491923641703378
Profundidad: 10 | F1: 0.5441988950276243
Profundidad: 11 | F1: 0.5401662049861496
Profundidad: 12 | F1: 0.549865229110512
Profundidad: 13 | F1: 0.5423728813559321
Profundidad: 14 | F1: 0.5031446540880503
Profundidad: 15 | F1: 0.5233881163084704
Profundidad: 16 | F1: 0.5116851168511685
Profundidad: 17 | F1: 0.5389369592088998
Profundidad: 18 | F1: 0.5224787363304981
Profundidad: 19 | F1: 0.5186972255729795
Profundidad: 20 | F1: 0.5011933174224343

Mejor profundidad: 7
Mejor F1: 0.5924855491329479


### Ajuste de profundidad del árbol de decisión

Se probaron valores de `max_depth` entre 1 y 20 para encontrar la profundidad
que maximiza F1 en el conjunto de validación.

El mejor resultado fue:

- `max_depth = 7`
- F1 ≈ 0.592

Este resultado supera ligeramente el objetivo mínimo de F1 = 0.59.

A partir de profundidades mayores, el F1 de validación dejó de mejorar y
comenzó a disminuir. Esto indica que aumentar la complejidad del árbol ya no
mejora su capacidad de generalización y puede ser una señal de sobreajuste.

### Bosque aleatorio

A continuación se evaluará un `RandomForestClassifier`.

Este modelo combina múltiples árboles de decisión, lo que normalmente permite
obtener predicciones más estables que utilizando un único árbol.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=7,
    random_state=12345
)

forest_model.fit(features_train, target_train)

forest_predicted_valid = forest_model.predict(features_valid)

forest_probabilities_valid = forest_model.predict_proba(features_valid)
forest_probabilities_one_valid = forest_probabilities_valid[:, 1]

forest_f1 = f1_score(
    target_valid,
    forest_predicted_valid
)

forest_auc_roc = roc_auc_score(
    target_valid,
    forest_probabilities_one_valid
)

print('F1 bosque aleatorio:', forest_f1)
print('AUC-ROC bosque aleatorio:', forest_auc_roc)

F1 bosque aleatorio: 0.5705329153605015
AUC-ROC bosque aleatorio: 0.8729077372154891


In [ ]:
best_depth_forest = 0
best_f1_forest = 0

for depth in range(1, 21):
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=depth,
        random_state=12345
    )

    model.fit(features_train, target_train)

    predicted_valid = model.predict(features_valid)

    f1 = f1_score(
        target_valid,
        predicted_valid
    )

    print('Profundidad:', depth, '| F1:', f1)

    if f1 > best_f1_forest:
        best_f1_forest = f1
        best_depth_forest = depth

print()
print('Mejor profundidad del bosque:', best_depth_forest)
print('Mejor F1 del bosque:', best_f1_forest)

Profundidad: 1 | F1: 0.0
Profundidad: 2 | F1: 0.1580135440180587
Profundidad: 3 | F1: 0.24733475479744135
Profundidad: 4 | F1: 0.4705882352941177
Profundidad: 5 | F1: 0.5201342281879194
Profundidad: 6 | F1: 0.5439739413680781
Profundidad: 7 | F1: 0.5705329153605015
Profundidad: 8 | F1: 0.5816485225505443
Profundidad: 9 | F1: 0.588597842835131
Profundidad: 10 | F1: 0.5929339477726575
Profundidad: 11 | F1: 0.6045801526717557
Profundidad: 12 | F1: 0.5984962406015037
Profundidad: 13 | F1: 0.6
Profundidad: 14 | F1: 0.6071428571428571
Profundidad: 15 | F1: 0.6151560178306092
Profundidad: 16 | F1: 0.6041666666666666
Profundidad: 17 | F1: 0.60932944606414
Profundidad: 18 | F1: 0.6120058565153733
Profundidad: 19 | F1: 0.5991058122205662
Profundidad: 20 | F1: 0.5949177877428998

Mejor profundidad del bosque: 15
Mejor F1 del bosque: 0.6151560178306092


### Ajuste de profundidad del bosque aleatorio

Se probaron valores de `max_depth` entre 1 y 20 utilizando un bosque aleatorio
con 100 árboles.

El mejor resultado en el conjunto de validación se obtuvo con:

- `n_estimators = 100`
- `max_depth = 15`
- F1 ≈ 0.615

Este resultado supera el objetivo mínimo de F1 = 0.59 y también mejora el mejor
resultado obtenido con un árbol de decisión individual.

Por lo tanto, el bosque aleatorio se considera actualmente el mejor modelo
evaluado.

In [ ]:
best_forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=12345
)

best_forest.fit(features_train, target_train)

predicted_valid_best_forest = best_forest.predict(features_valid)

probabilities_valid_best_forest = best_forest.predict_proba(features_valid)
probabilities_one_valid_best_forest = probabilities_valid_best_forest[:, 1]

f1_best_forest = f1_score(
    target_valid,
    predicted_valid_best_forest
)

auc_roc_best_forest = roc_auc_score(
    target_valid,
    probabilities_one_valid_best_forest
)

print('F1 mejor bosque:', f1_best_forest)
print('AUC-ROC mejor bosque:', auc_roc_best_forest)

F1 mejor bosque: 0.6151560178306092
AUC-ROC mejor bosque: 0.8661059587151444


In [ ]:
balanced_forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=12345,
    class_weight='balanced'
)

balanced_forest.fit(features_train, target_train)

predicted_valid_balanced_forest = balanced_forest.predict(features_valid)

probabilities_valid_balanced_forest = balanced_forest.predict_proba(features_valid)
probabilities_one_valid_balanced_forest = probabilities_valid_balanced_forest[:, 1]

f1_balanced_forest = f1_score(
    target_valid,
    predicted_valid_balanced_forest
)

auc_roc_balanced_forest = roc_auc_score(
    target_valid,
    probabilities_one_valid_balanced_forest
)

print('F1 bosque balanceado:', f1_balanced_forest)
print('AUC-ROC bosque balanceado:', auc_roc_balanced_forest)

F1 bosque balanceado: 0.5988372093023255
AUC-ROC bosque balanceado: 0.8624140925214308


### Bosque aleatorio con pesos de clase

También se evaluó el bosque aleatorio utilizando
`class_weight='balanced'`.

Los resultados fueron aproximadamente:

- F1 = 0.599
- AUC-ROC = 0.862

El modelo mantiene un F1 superior al mínimo requerido, pero su rendimiento es
inferior al bosque aleatorio sin balanceo:

- Bosque aleatorio sin balanceo: F1 ≈ 0.615
- Bosque aleatorio con pesos de clase: F1 ≈ 0.599

Por el momento, el bosque sin balanceo presenta un mejor resultado en
validación. Sin embargo, también se evaluará un bosque entrenado con
sobremuestreo antes de seleccionar el modelo final.

### Bosque aleatorio con sobremuestreo

Como última configuración, se entrenará un bosque aleatorio utilizando el
conjunto de entrenamiento balanceado mediante sobremuestreo.

Se probarán distintas profundidades utilizando únicamente el conjunto de
validación para seleccionar la mejor configuración.

In [ ]:
best_depth_upsampled_forest = 0
best_f1_upsampled_forest = 0

for depth in range(5, 21):
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=depth,
        random_state=12345
    )

    model.fit(
        features_upsampled,
        target_upsampled
    )

    predicted_valid = model.predict(features_valid)

    f1 = f1_score(
        target_valid,
        predicted_valid
    )

    print('Profundidad:', depth, '| F1:', f1)

    if f1 > best_f1_upsampled_forest:
        best_f1_upsampled_forest = f1
        best_depth_upsampled_forest = depth

print()
print('Mejor profundidad:', best_depth_upsampled_forest)
print('Mejor F1:', best_f1_upsampled_forest)

Profundidad: 5 | F1: 0.6165556612749763
Profundidad: 6 | F1: 0.6267190569744597
Profundidad: 7 | F1: 0.6348884381338742
Profundidad: 8 | F1: 0.6376811594202898
Profundidad: 9 | F1: 0.6374331550802138
Profundidad: 10 | F1: 0.6351648351648351
Profundidad: 11 | F1: 0.6436781609195401
Profundidad: 12 | F1: 0.63689776733255
Profundidad: 13 | F1: 0.6341463414634146
Profundidad: 14 | F1: 0.6323529411764706
Profundidad: 15 | F1: 0.6269035532994924
Profundidad: 16 | F1: 0.6290115532734274
Profundidad: 17 | F1: 0.6253229974160207
Profundidad: 18 | F1: 0.6133682830930538
Profundidad: 19 | F1: 0.623342175066313
Profundidad: 20 | F1: 0.6261558784676354

Mejor profundidad: 11
Mejor F1: 0.6436781609195401


In [ ]:
best_upsampled_forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=11,
    random_state=12345
)

best_upsampled_forest.fit(
    features_upsampled,
    target_upsampled
)

predicted_valid_upsampled_forest = best_upsampled_forest.predict(features_valid)

probabilities_valid_upsampled_forest = best_upsampled_forest.predict_proba(
    features_valid
)[:, 1]

f1_valid_upsampled_forest = f1_score(
    target_valid,
    predicted_valid_upsampled_forest
)

auc_roc_valid_upsampled_forest = roc_auc_score(
    target_valid,
    probabilities_valid_upsampled_forest
)

print('F1 validación:', f1_valid_upsampled_forest)
print('AUC-ROC validación:', auc_roc_valid_upsampled_forest)

F1 validación: 0.6436781609195401
AUC-ROC validación: 0.8668988323972805


### Selección del mejor modelo

El bosque aleatorio entrenado con sobremuestreo obtuvo el mejor resultado en el
conjunto de validación.

La configuración seleccionada fue:

- `n_estimators = 100`
- `max_depth = 11`
- entrenamiento con sobremuestreo.

Resultados en validación:

- F1 ≈ 0.644
- AUC-ROC ≈ 0.867

Este modelo supera el objetivo mínimo de F1 = 0.59 y obtiene el mayor F1 de
todos los modelos evaluados.

Por este motivo, se selecciona como modelo final y será evaluado una única vez
sobre el conjunto de prueba.

## Evaluación final

Una vez seleccionado el mejor modelo utilizando el conjunto de validación, se
evalúa su rendimiento final sobre el conjunto de prueba.

El conjunto de prueba no se utiliza para seleccionar modelos ni ajustar
hiperparámetros.

In [ ]:
predicted_test_upsampled_forest = best_upsampled_forest.predict(features_test)

probabilities_test_upsampled_forest = best_upsampled_forest.predict_proba(
    features_test
)[:, 1]

f1_test_upsampled_forest = f1_score(
    target_test,
    predicted_test_upsampled_forest
)

auc_roc_test_upsampled_forest = roc_auc_score(
    target_test,
    probabilities_test_upsampled_forest
)

print('F1 test:', f1_test_upsampled_forest)
print('AUC-ROC test:', auc_roc_test_upsampled_forest)

F1 test: 0.5900116144018583
AUC-ROC test: 0.8506272065594098


In [ ]:
best_tree = DecisionTreeClassifier(
    max_depth=7,
    random_state=12345
)

best_tree.fit(features_train, target_train)

tree_probabilities_valid_best = best_tree.predict_proba(features_valid)[:, 1]

tree_auc_roc_best = roc_auc_score(
    target_valid,
    tree_probabilities_valid_best
)

print('AUC-ROC árbol max_depth=7:', tree_auc_roc_best)

AUC-ROC árbol max_depth=7: 0.8230390617302197


## Comparación de modelos

| Modelo | Tratamiento del desequilibrio | F1 validación | AUC-ROC validación |
|---|---|---:|---:|
| Regresión logística | Ninguno | 0.308 | 0.788 |
| Regresión logística | `class_weight='balanced'` | 0.529 | 0.794 |
| Regresión logística | Sobremuestreo | 0.525 | 0.794 |
| Árbol de decisión | `max_depth=7` | 0.592 | 0.823 |
| Bosque aleatorio | `max_depth=15` | 0.615 | 0.866 |
| Bosque aleatorio | `class_weight='balanced'` | 0.599 | 0.862 |
| Bosque aleatorio | Sobremuestreo, `max_depth=11` | **0.644** | **0.867** |

### Resultado final en prueba

| Métrica | Resultado |
|---|---:|
| F1 | **0.590** |
| AUC-ROC | **0.851** |

### Comparación entre F1 y AUC-ROC

F1 y AUC-ROC evalúan aspectos diferentes del rendimiento del modelo.

F1 combina `precision` y `recall` para evaluar qué tan bien el modelo identifica
la clase positiva (`Exited = 1`) con una decisión de clasificación concreta.

AUC-ROC, en cambio, mide la capacidad general del modelo para distinguir entre
las dos clases considerando diferentes umbrales de clasificación.

El modelo final obtuvo:

- F1 ≈ 0.590
- AUC-ROC ≈ 0.851

El valor F1 cumple el objetivo mínimo del proyecto. El AUC-ROC, además, muestra
que el modelo tiene una buena capacidad para separar clientes que abandonan el
banco de aquellos que permanecen.

# Conclusión final

El objetivo del proyecto fue desarrollar un modelo capaz de predecir si un
cliente de Beta Bank abandonará el banco.

Durante la exploración de los datos se identificó un desequilibrio importante
en la variable objetivo `Exited`: aproximadamente el 79.6% de los clientes
permanece en el banco y el 20.4% abandona.

También se encontraron 909 valores ausentes en `Tenure`. Estos valores fueron
imputados utilizando la mediana calculada únicamente con el conjunto de
entrenamiento.

Las columnas `RowNumber`, `CustomerId` y `Surname` fueron eliminadas por no
representar características relevantes para la predicción. Las variables
categóricas fueron transformadas mediante One-Hot Encoding y las variables
numéricas fueron estandarizadas.

Los datos se dividieron en:

- 60% entrenamiento;
- 20% validación;
- 20% prueba.

La división se realizó utilizando estratificación para conservar la proporción
de clases.

Se evaluaron distintos modelos y estrategias de balanceo:

- regresión logística sin balanceo;
- regresión logística con `class_weight='balanced'`;
- regresión logística con sobremuestreo;
- árbol de decisión;
- bosque aleatorio;
- bosque aleatorio con pesos de clase;
- bosque aleatorio con sobremuestreo.

El mejor resultado de validación se obtuvo con un
`RandomForestClassifier` entrenado con sobremuestreo y los siguientes
hiperparámetros:

- `n_estimators = 100`;
- `max_depth = 11`;
- `random_state = 12345`.

Resultados de validación:

- F1 ≈ 0.644;
- AUC-ROC ≈ 0.867.

Finalmente, el modelo seleccionado fue evaluado sobre el conjunto de prueba y
obtuvo:

- F1 ≈ 0.590;
- AUC-ROC ≈ 0.851.

El F1 cumple el requisito mínimo de 0.59 establecido para el proyecto.

El AUC-ROC indica además una buena capacidad del modelo para distinguir entre
clientes que abandonan el banco y clientes que permanecen.

Por lo tanto, el bosque aleatorio entrenado con sobremuestreo se selecciona
como modelo final.